In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv()

df.columns = (
    df.columns.str.strip()
    .str.replace(r"[^A-Za-z0-9]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)

print("First rows:")
print(df.head())

First rows:
   customerid  gender  seniorcitizen partner dependents  tenure phoneservice  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      multiplelines internetservice onlinesecurity  ... deviceprotection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  techsupport streamingtv streamingmovies        c

In [3]:
print("\nDataset information:")
df.info()


Dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerid        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   seniorcitizen     7043 non-null   int64  
 3   partner           7043 non-null   str    
 4   dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   phoneservice      7043 non-null   str    
 7   multiplelines     7043 non-null   str    
 8   internetservice   7043 non-null   str    
 9   onlinesecurity    7043 non-null   str    
 10  onlinebackup      7043 non-null   str    
 11  deviceprotection  7043 non-null   str    
 12  techsupport       7043 non-null   str    
 13  streamingtv       7043 non-null   str    
 14  streamingmovies   7043 non-null   str    
 15  contract          7043 non-null   str    
 16  paperlessbilling  7043 non-null

In [4]:
print("\nMissing values before cleaning:")
print(df.isna().sum())


Missing values before cleaning:
customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64


In [5]:
df = df.drop_duplicates().copy()


In [6]:
for col in ["tenure", "monthlycharges", "totalcharges"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

categorical_cols = df.select_dtypes(include=["object", "string"]).columns

for col in categorical_cols:
    if col != "customerid":
        df[col] = (
            df[col].astype("string")
            .str.strip()
            .str.lower()
            .replace("", pd.NA)
        )


In [7]:
if {"tenure", "totalcharges"}.issubset(df.columns):
    no_tenure = df["tenure"].eq(0) & df["totalcharges"].isna()
    df.loc[no_tenure, "totalcharges"] = 0


numeric_cols = df.select_dtypes(include="number").columns

for col in numeric_cols:
    median_value = df[col].median()
    if pd.notna(median_value):
        df[col] = df[col].fillna(median_value)

In [8]:
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

for col in categorical_cols:
    if col != "customerid":
        mode_values = df[col].mode(dropna=True)
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

if {"tenure", "monthlycharges"}.issubset(df.columns):
    df["estimated_lifetime_charges"] = (
        df["tenure"] * df["monthlycharges"]
    )

In [9]:
continuous_cols = [
    col for col in ["tenure", "monthlycharges", "totalcharges"]
    if col in df.columns
]

for col in continuous_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    df[f"{col}_outlier_flag"] = (
        (df[col] < q1 - 1.5 * iqr)
        | (df[col] > q3 + 1.5 * iqr)
    )

In [10]:
print("\nCleaned shape:", df.shape)
print("\nMissing values after cleaning:")
print(df.isna().sum())


Cleaned shape: (7043, 25)

Missing values after cleaning:
customerid                     0
gender                         0
seniorcitizen                  0
partner                        0
dependents                     0
tenure                         0
phoneservice                   0
multiplelines                  0
internetservice                0
onlinesecurity                 0
onlinebackup                   0
deviceprotection               0
techsupport                    0
streamingtv                    0
streamingmovies                0
contract                       0
paperlessbilling               0
paymentmethod                  0
monthlycharges                 0
totalcharges                   0
churn                          0
estimated_lifetime_charges     0
tenure_outlier_flag            0
monthlycharges_outlier_flag    0
totalcharges_outlier_flag      0
dtype: int64


In [11]:
print("\nOutlier flag counts:")
for col in continuous_cols:
    flag_col = f"{col}_outlier_flag"
    print(f"{flag_col}:", df[flag_col].sum())

df.to_csv("Telecom_Customer_Churn_Cleaned.csv", index=False)
print("\nCleaned dataset saved as Telecom_Customer_Churn_Cleaned.csv")




Outlier flag counts:
tenure_outlier_flag: 0
monthlycharges_outlier_flag: 0
totalcharges_outlier_flag: 0

Cleaned dataset saved as Telecom_Customer_Churn_Cleaned.csv
